# EMP Training vs Attention Sink Emergence

This notebook investigates whether applying EMP (Effective Model Pruning) to attention scores **during training** can prevent or reduce the emergence of **attention sinks**.

## Background

**Attention Sink** (Xiao et al., 2023; Gu et al., 2024):
- First tokens in autoregressive LMs receive disproportionately high attention
- Caused by softmax normalization forcing attention to sum to 1
- Model uses first tokens as "garbage collectors" for unused attention mass
- Emerges after ~1k-2k training steps

**Key insight from "When Attention Sink Emerges" (ICLR 2025)**:
- Sigmoid attention (no normalization) eliminates attention sinks
- Attention sink stems from "tokens' inner dependence on attention scores"

**Our Hypothesis**:
EMP during training prunes low-importance attention scores → model cannot "dump" attention on sink tokens → learns more meaningful attention patterns.

## Experimental Design

1. **Baseline**: Train normal transformer, observe attention sink emergence
2. **EMP Training**: Train with EMP applied to pre-softmax attention scores
3. **Gated Attention (optional)**: Train with learnable key biases as control

**Metrics**:
- Sink strength: attention weight allocated to first k tokens
- N_eff distribution evolution during training
- Perplexity comparison
- Attention entropy

## 1. Setup

In [ ]:
import os
import math
import time
import copy
import json
from dataclasses import dataclass, field
from typing import Optional, Tuple, Dict, List, Literal
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

import numpy as np

# For WikiText-2 dataset
try:
    from datasets import load_dataset
except ImportError:
    print("Installing datasets library...")
    !pip install datasets -q
    from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
@dataclass
class TransformerConfig:
    """Configuration for GPT-style transformer."""
    vocab_size: int = 50257
    block_size: int = 256       # context length
    n_layer: int = 6            # number of transformer blocks
    n_head: int = 8             # number of attention heads
    n_embd: int = 512           # embedding dimension
    dropout: float = 0.1
    bias: bool = True
    
    # Training config
    batch_size: int = 32
    total_steps: int = 5000     # Total training steps
    lr: float = 3e-4
    weight_decay: float = 0.1   # Higher weight decay encourages attention sink
    warmup_steps: int = 100
    
    # EMP config
    emp_mode: Literal['none', 'per_head', 'global'] = 'none'
    emp_beta: float = 1.0
    
    # Attention type
    attention_type: Literal['softmax', 'sigmoid', 'softmax_emp'] = 'softmax'
    
    # Logging
    log_interval: int = 100     # Log metrics every N steps
    eval_interval: int = 500    # Evaluate every N steps

# Create configs for different experiments
def create_baseline_config():
    return TransformerConfig(
        attention_type='softmax',
        emp_mode='none'
    )

def create_emp_config(beta=1.0, mode='per_head'):
    return TransformerConfig(
        attention_type='softmax_emp',
        emp_mode=mode,
        emp_beta=beta
    )

def create_sigmoid_config():
    """Sigmoid attention without normalization (no attention sink expected)."""
    return TransformerConfig(
        attention_type='sigmoid',
        emp_mode='none'
    )

print("Configurations ready.")

## 3. Dataset

In [ ]:
class WordTokenizer:
    """Simple word-level tokenizer."""
    def __init__(self, text: str, max_vocab: int = 20000):
        words = text.split()
        word_freq = {}
        for w in words:
            word_freq[w] = word_freq.get(w, 0) + 1
        
        sorted_words = sorted(word_freq.items(), key=lambda x: -x[1])
        vocab_words = [w for w, _ in sorted_words[:max_vocab - 2]]
        
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
        
        all_tokens = [self.pad_token, self.unk_token] + vocab_words
        self.word_to_idx = {w: i for i, w in enumerate(all_tokens)}
        self.idx_to_word = {i: w for i, w in enumerate(all_tokens)}
        self.vocab_size = len(all_tokens)
        self.unk_idx = self.word_to_idx[self.unk_token]
    
    def encode(self, text: str) -> List[int]:
        return [self.word_to_idx.get(w, self.unk_idx) for w in text.split()]
    
    def decode(self, indices: List[int]) -> str:
        return ' '.join([self.idx_to_word.get(i, self.unk_token) for i in indices])

In [ ]:
class LMDataset(Dataset):
    """Language Modeling Dataset."""
    def __init__(self, data: torch.Tensor, block_size: int):
        self.data = data
        self.block_size = block_size
    
    def __len__(self):
        return max(0, len(self.data) - self.block_size - 1)
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]
        return x, y


def load_wikitext2(config: TransformerConfig):
    """Load WikiText-2 dataset."""
    print("Loading WikiText-2...")
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
    
    train_text = '\n'.join(dataset['train']['text'])
    val_text = '\n'.join(dataset['validation']['text'])
    
    tokenizer = WordTokenizer(train_text, max_vocab=20000)
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    
    train_ids = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
    val_ids = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
    
    print(f"Train tokens: {len(train_ids):,}")
    print(f"Val tokens: {len(val_ids):,}")
    
    train_dataset = LMDataset(train_ids, config.block_size)
    val_dataset = LMDataset(val_ids, config.block_size)
    
    # num_workers=0 for Windows compatibility
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, 
                              num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, 
                            num_workers=0, pin_memory=torch.cuda.is_available())
    
    return train_loader, val_loader, tokenizer

In [ ]:
# Load data once
base_config = TransformerConfig()
train_loader, val_loader, tokenizer = load_wikitext2(base_config)
VOCAB_SIZE = tokenizer.vocab_size
print(f"\nVocab size: {VOCAB_SIZE}")

## 4. Transformer Model with Multiple Attention Types

In [ ]:
def compute_sink_metrics(attn_weights: torch.Tensor, k: int = 4) -> Dict[str, float]:
    """
    Compute attention sink metrics from attention weights.
    
    Args:
        attn_weights: Post-softmax attention weights (B, H, T, T)
        k: Number of initial tokens to consider as potential sinks
    
    Returns:
        Dictionary with sink metrics
    """
    B, H, T, _ = attn_weights.shape
    
    # Sink strength: average attention to first k tokens
    # For each query position, what fraction of attention goes to first k tokens?
    sink_attn = attn_weights[:, :, :, :k].sum(dim=-1)  # (B, H, T)
    
    # Average across all positions (excluding first k to avoid self-attention bias)
    if T > k:
        sink_strength = sink_attn[:, :, k:].mean().item()
    else:
        sink_strength = sink_attn.mean().item()
    
    # First token sink: attention specifically to position 0
    first_token_attn = attn_weights[:, :, :, 0]  # (B, H, T)
    if T > 1:
        first_token_sink = first_token_attn[:, :, 1:].mean().item()
    else:
        first_token_sink = first_token_attn.mean().item()
    
    # Attention entropy (higher = more uniform)
    # Only compute for valid (non-zero) attention weights
    eps = 1e-10
    log_attn = torch.log(attn_weights + eps)
    entropy = -(attn_weights * log_attn).sum(dim=-1)  # (B, H, T)
    avg_entropy = entropy.mean().item()
    
    # Max attention per query (lower = more distributed)
    max_attn = attn_weights.max(dim=-1)[0].mean().item()
    
    return {
        'sink_strength_k': sink_strength,
        'first_token_sink': first_token_sink,
        'attention_entropy': avg_entropy,
        'max_attention': max_attn,
    }


def compute_neff(scores: torch.Tensor) -> float:
    """Compute N_eff from attention scores."""
    s = scores.reshape(-1)
    s_abs = torch.abs(s)
    s_sum = s_abs.sum()
    if s_sum < 1e-10:
        return float(len(s))
    omega = s_abs / s_sum
    neff = 1.0 / (omega ** 2).sum()
    return neff.item()

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention with multiple attention types:
    - softmax: Standard softmax attention (baseline, will have attention sinks)
    - sigmoid: Sigmoid attention without normalization (no attention sink expected)
    - softmax_emp: Softmax with EMP applied to pre-softmax scores
    """
    
    def __init__(self, config: TransformerConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.dropout = config.dropout
        self.attention_type = config.attention_type
        self.emp_mode = config.emp_mode
        self.emp_beta = config.emp_beta
        
        # QKV projection
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        # Causal mask
        self.register_buffer("mask", torch.tril(torch.ones(config.block_size, config.block_size))
                                          .view(1, 1, config.block_size, config.block_size))
        
        # Storage for analysis
        self.last_attn_weights = None  # Post-softmax/sigmoid weights
        self.last_attn_scores = None   # Pre-softmax scores
        self.last_emp_mask = None
        self.last_neff = None
    
    def apply_emp_mask(self, scores: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, float]:
        """Apply EMP-based masking to attention scores."""
        B, H, T, T2 = scores.shape
        
        if self.emp_mode == 'per_head':
            masks = []
            neffs = []
            for h in range(H):
                head_scores = scores[:, h, :, :]
                # Only consider valid (non -inf) scores
                valid_mask = head_scores != float('-inf')
                valid_scores = head_scores[valid_mask]
                
                if valid_scores.numel() == 0:
                    head_mask = torch.ones_like(head_scores, dtype=torch.bool)
                    neffs.append(0)
                else:
                    neff = compute_neff(valid_scores)
                    neffs.append(neff)
                    r_neff = max(1, int(self.emp_beta * neff))
                    r_neff = min(r_neff, valid_scores.numel())
                    
                    if r_neff >= valid_scores.numel():
                        head_mask = torch.ones_like(head_scores, dtype=torch.bool)
                    else:
                        # Find threshold based on valid scores only
                        flat_valid = valid_scores.reshape(-1)
                        _, indices = torch.sort(torch.abs(flat_valid), descending=True)
                        thresh = torch.abs(flat_valid)[indices[r_neff - 1]]
                        head_mask = (torch.abs(head_scores) >= thresh) | (~valid_mask)
                
                masks.append(head_mask.unsqueeze(1))
            
            mask = torch.cat(masks, dim=1)
            avg_neff = sum(neffs) / len(neffs) if neffs else 0
        
        elif self.emp_mode == 'global':
            valid_mask = scores != float('-inf')
            valid_scores = scores[valid_mask]
            
            if valid_scores.numel() == 0:
                mask = torch.ones_like(scores, dtype=torch.bool)
                avg_neff = 0
            else:
                avg_neff = compute_neff(valid_scores)
                r_neff = max(1, int(self.emp_beta * avg_neff))
                r_neff = min(r_neff, valid_scores.numel())
                
                if r_neff >= valid_scores.numel():
                    mask = torch.ones_like(scores, dtype=torch.bool)
                else:
                    flat_valid = valid_scores.reshape(-1)
                    _, indices = torch.sort(torch.abs(flat_valid), descending=True)
                    thresh = torch.abs(flat_valid)[indices[r_neff - 1]]
                    mask = (torch.abs(scores) >= thresh) | (~valid_mask)
        else:
            mask = torch.ones_like(scores, dtype=torch.bool)
            avg_neff = 0
        
        # Apply mask
        masked_scores = scores.clone()
        masked_scores[~mask] = float('-inf')
        
        return masked_scores, mask, avg_neff
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        
        # QKV projection
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        # Reshape for multi-head attention
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        
        # Apply causal mask
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        
        # Store pre-softmax scores
        self.last_attn_scores = scores.detach()
        
        # Apply attention based on type
        if self.attention_type == 'softmax':
            attn_weights = F.softmax(scores, dim=-1)
            self.last_emp_mask = None
            self.last_neff = None
            
        elif self.attention_type == 'sigmoid':
            # Sigmoid attention without normalization (no attention sink expected)
            # Mask out future positions with 0 instead of -inf
            causal_mask = self.mask[:, :, :T, :T]
            attn_weights = torch.sigmoid(scores) * causal_mask
            self.last_emp_mask = None
            self.last_neff = None
            
        elif self.attention_type == 'softmax_emp':
            # Apply EMP before softmax
            scores, emp_mask, neff = self.apply_emp_mask(scores)
            attn_weights = F.softmax(scores, dim=-1)
            self.last_emp_mask = emp_mask
            self.last_neff = neff
        
        else:
            raise ValueError(f"Unknown attention type: {self.attention_type}")
        
        # Handle NaN from all -inf rows
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        
        # Store post-softmax weights for analysis
        self.last_attn_weights = attn_weights.detach()
        
        # Dropout and apply attention
        attn_weights = self.attn_dropout(attn_weights)
        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.c_proj(out))
        
        return out

In [ ]:
class MLP(nn.Module):
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model parameters: {n_params/1e6:.2f}M")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None):
        B, T = idx.shape
        
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss
    
    def get_attention_metrics(self) -> Dict[str, float]:
        """Aggregate attention metrics from all layers."""
        all_metrics = defaultdict(list)
        
        for i, block in enumerate(self.transformer.h):
            attn = block.attn
            if attn.last_attn_weights is not None:
                metrics = compute_sink_metrics(attn.last_attn_weights)
                for k, v in metrics.items():
                    all_metrics[f'{k}'].append(v)
                
                if attn.last_neff is not None:
                    all_metrics['neff'].append(attn.last_neff)
        
        # Average across layers
        avg_metrics = {}
        for k, v in all_metrics.items():
            avg_metrics[k] = sum(v) / len(v) if v else 0
        
        return avg_metrics

## 5. Training Loop with Metric Logging

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, max_batches: int = 50) -> Tuple[float, float]:
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    
    avg_loss = total_loss / total_tokens
    return avg_loss, math.exp(avg_loss)


def train_model(config: TransformerConfig, train_loader: DataLoader, val_loader: DataLoader,
                experiment_name: str) -> Dict:
    """
    Train model and log attention sink metrics throughout training.
    
    Returns:
        Dictionary with training history and final metrics
    """
    config.vocab_size = VOCAB_SIZE
    model = GPT(config).to(device)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95)
    )
    
    # Linear warmup + cosine decay
    def lr_lambda(step):
        if step < config.warmup_steps:
            return step / config.warmup_steps
        else:
            progress = (step - config.warmup_steps) / (config.total_steps - config.warmup_steps)
            return 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    # History tracking
    history = {
        'steps': [],
        'train_loss': [],
        'val_loss': [],
        'val_ppl': [],
        'sink_strength': [],
        'first_token_sink': [],
        'attention_entropy': [],
        'max_attention': [],
        'neff': [],
    }
    
    print(f"\n{'='*80}")
    print(f"Training: {experiment_name}")
    print(f"Attention type: {config.attention_type}, EMP mode: {config.emp_mode}, beta: {config.emp_beta}")
    print(f"{'='*80}")
    
    step = 0
    train_iter = iter(train_loader)
    running_loss = 0.0
    
    model.train()
    start_time = time.time()
    
    while step < config.total_steps:
        # Get batch (with wraparound)
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)
        
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        step += 1
        
        # Log metrics
        if step % config.log_interval == 0:
            avg_loss = running_loss / config.log_interval
            running_loss = 0.0
            
            # Get attention metrics
            attn_metrics = model.get_attention_metrics()
            
            history['steps'].append(step)
            history['train_loss'].append(avg_loss)
            history['sink_strength'].append(attn_metrics.get('sink_strength_k', 0))
            history['first_token_sink'].append(attn_metrics.get('first_token_sink', 0))
            history['attention_entropy'].append(attn_metrics.get('attention_entropy', 0))
            history['max_attention'].append(attn_metrics.get('max_attention', 0))
            history['neff'].append(attn_metrics.get('neff', 0))
            
            elapsed = time.time() - start_time
            print(f"Step {step:5d} | Loss: {avg_loss:.4f} | "
                  f"Sink: {attn_metrics.get('first_token_sink', 0):.4f} | "
                  f"Entropy: {attn_metrics.get('attention_entropy', 0):.3f} | "
                  f"Time: {elapsed:.1f}s")
        
        # Evaluate
        if step % config.eval_interval == 0:
            val_loss, val_ppl = evaluate(model, val_loader)
            history['val_loss'].append(val_loss)
            history['val_ppl'].append(val_ppl)
            print(f"  -> Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")
            model.train()
    
    # Final evaluation
    val_loss, val_ppl = evaluate(model, val_loader, max_batches=100)
    
    print(f"\nFinal - Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")
    print(f"Final Sink Strength: {history['first_token_sink'][-1]:.4f}")
    
    return {
        'config': config,
        'history': history,
        'final_val_loss': val_loss,
        'final_val_ppl': val_ppl,
        'model': model,
    }

## 6. Run Experiments

In [ ]:
# Store all results
all_results = {}

# Experiment 1: Baseline (standard softmax attention)
print("\n" + "#"*80)
print("# EXPERIMENT 1: BASELINE (Softmax Attention)")
print("#"*80)

baseline_config = create_baseline_config()
baseline_results = train_model(baseline_config, train_loader, val_loader, "Baseline")
all_results['baseline'] = baseline_results

In [ ]:
# Experiment 2: Sigmoid attention (control - no attention sink expected)
print("\n" + "#"*80)
print("# EXPERIMENT 2: SIGMOID ATTENTION (No Normalization)")
print("#"*80)

sigmoid_config = create_sigmoid_config()
sigmoid_results = train_model(sigmoid_config, train_loader, val_loader, "Sigmoid")
all_results['sigmoid'] = sigmoid_results

In [ ]:
# Experiment 3: EMP Training (per-head, beta=1.0)
print("\n" + "#"*80)
print("# EXPERIMENT 3: EMP TRAINING (Per-Head, beta=1.0)")
print("#"*80)

emp_config = create_emp_config(beta=1.0, mode='per_head')
emp_results = train_model(emp_config, train_loader, val_loader, "EMP_beta1.0")
all_results['emp_beta1.0'] = emp_results

In [ ]:
# Experiment 4: EMP Training (more aggressive, beta=0.5)
print("\n" + "#"*80)
print("# EXPERIMENT 4: EMP TRAINING (Per-Head, beta=0.5)")
print("#"*80)

emp_aggressive_config = create_emp_config(beta=0.5, mode='per_head')
emp_aggressive_results = train_model(emp_aggressive_config, train_loader, val_loader, "EMP_beta0.5")
all_results['emp_beta0.5'] = emp_aggressive_results

In [ ]:
# Experiment 5: EMP Training (mild, beta=1.5)
print("\n" + "#"*80)
print("# EXPERIMENT 5: EMP TRAINING (Per-Head, beta=1.5)")
print("#"*80)

emp_mild_config = create_emp_config(beta=1.5, mode='per_head')
emp_mild_results = train_model(emp_mild_config, train_loader, val_loader, "EMP_beta1.5")
all_results['emp_beta1.5'] = emp_mild_results

## 7. Analysis and Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    colors = {
        'baseline': 'blue',
        'sigmoid': 'green',
        'emp_beta0.5': 'red',
        'emp_beta1.0': 'orange',
        'emp_beta1.5': 'purple',
    }
    
    # Plot 1: Training Loss
    ax = axes[0, 0]
    for name, result in all_results.items():
        h = result['history']
        ax.plot(h['steps'], h['train_loss'], label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Step')
    ax.set_ylabel('Training Loss')
    ax.set_title('Training Loss over Steps')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: First Token Sink Strength
    ax = axes[0, 1]
    for name, result in all_results.items():
        h = result['history']
        ax.plot(h['steps'], h['first_token_sink'], label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Step')
    ax.set_ylabel('First Token Attention')
    ax.set_title('Attention Sink Strength (First Token)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Sink Strength (k tokens)
    ax = axes[0, 2]
    for name, result in all_results.items():
        h = result['history']
        ax.plot(h['steps'], h['sink_strength'], label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Step')
    ax.set_ylabel('Sink Strength (k=4)')
    ax.set_title('Attention to First k Tokens')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Attention Entropy
    ax = axes[1, 0]
    for name, result in all_results.items():
        h = result['history']
        ax.plot(h['steps'], h['attention_entropy'], label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Step')
    ax.set_ylabel('Attention Entropy')
    ax.set_title('Attention Entropy (Higher = More Uniform)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 5: Max Attention
    ax = axes[1, 1]
    for name, result in all_results.items():
        h = result['history']
        ax.plot(h['steps'], h['max_attention'], label=name, color=colors.get(name, 'gray'))
    ax.set_xlabel('Step')
    ax.set_ylabel('Max Attention')
    ax.set_title('Max Attention per Query (Lower = More Distributed)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 6: Final Performance Comparison
    ax = axes[1, 2]
    names = list(all_results.keys())
    ppls = [all_results[n]['final_val_ppl'] for n in names]
    sinks = [all_results[n]['history']['first_token_sink'][-1] for n in names]
    
    x = np.arange(len(names))
    width = 0.35
    
    ax2 = ax.twinx()
    bars1 = ax.bar(x - width/2, ppls, width, label='Perplexity', color='steelblue')
    bars2 = ax2.bar(x + width/2, sinks, width, label='Sink Strength', color='coral')
    
    ax.set_xlabel('Experiment')
    ax.set_ylabel('Perplexity', color='steelblue')
    ax2.set_ylabel('Sink Strength', color='coral')
    ax.set_title('Final Performance vs Sink Strength')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    os.makedirs('./checkpoints', exist_ok=True)
    plt.savefig('./checkpoints/emp_attention_sink_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nPlot saved to ./checkpoints/emp_attention_sink_training.png")
    
except ImportError as e:
    print(f"Matplotlib not available: {e}")

In [ ]:
# Summary table
print("\n" + "="*100)
print("EXPERIMENT SUMMARY")
print("="*100)
print(f"{'Experiment':<20} | {'Val PPL':>10} | {'Sink (1st)':>12} | {'Sink (k=4)':>12} | {'Entropy':>10} | {'Max Attn':>10}")
print("-"*100)

for name, result in all_results.items():
    h = result['history']
    print(f"{name:<20} | {result['final_val_ppl']:>10.2f} | "
          f"{h['first_token_sink'][-1]:>12.4f} | "
          f"{h['sink_strength'][-1]:>12.4f} | "
          f"{h['attention_entropy'][-1]:>10.3f} | "
          f"{h['max_attention'][-1]:>10.4f}")

In [ ]:
def analyze_per_layer_sinks(model: nn.Module, loader: DataLoader, name: str):
    """Analyze sink strength per layer."""
    model.eval()
    
    x, y = next(iter(loader))
    x = x.to(device)
    
    with torch.no_grad():
        _ = model(x)
    
    print(f"\nPer-Layer Analysis: {name}")
    print("-"*60)
    
    for i, block in enumerate(model.transformer.h):
        attn = block.attn
        if attn.last_attn_weights is not None:
            metrics = compute_sink_metrics(attn.last_attn_weights)
            print(f"Layer {i}: Sink={metrics['first_token_sink']:.4f} | "
                  f"Entropy={metrics['attention_entropy']:.3f} | "
                  f"Max={metrics['max_attention']:.4f}")

# Analyze each model
for name, result in all_results.items():
    analyze_per_layer_sinks(result['model'], val_loader, name)

In [ ]:
def visualize_attention_pattern(model: nn.Module, loader: DataLoader, name: str, layer_idx: int = 0):
    """Visualize attention pattern for a specific layer."""
    model.eval()
    
    x, y = next(iter(loader))
    x = x.to(device)
    
    with torch.no_grad():
        _ = model(x)
    
    attn_weights = model.transformer.h[layer_idx].attn.last_attn_weights
    
    if attn_weights is None:
        print(f"No attention weights available for {name}")
        return
    
    # Average over batch and heads, take first 64 tokens
    attn_avg = attn_weights[0].mean(dim=0)[:64, :64].cpu().numpy()
    
    try:
        import matplotlib.pyplot as plt
        
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(attn_avg, cmap='viridis', aspect='auto')
        ax.set_xlabel('Key Position')
        ax.set_ylabel('Query Position')
        ax.set_title(f'Attention Pattern: {name} (Layer {layer_idx})')
        plt.colorbar(im, ax=ax, label='Attention Weight')
        
        # Highlight first column (attention sink)
        ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, alpha=0.7)
        
        plt.tight_layout()
        plt.savefig(f'./checkpoints/attention_pattern_{name}.png', dpi=150, bbox_inches='tight')
        plt.show()
    except ImportError:
        print("Matplotlib not available")

# Visualize attention for baseline and EMP
for name in ['baseline', 'emp_beta1.0', 'sigmoid']:
    if name in all_results:
        visualize_attention_pattern(all_results[name]['model'], val_loader, name, layer_idx=0)

In [ ]:
# Save results
results_to_save = {}
for name, result in all_results.items():
    results_to_save[name] = {
        'attention_type': result['config'].attention_type,
        'emp_mode': result['config'].emp_mode,
        'emp_beta': result['config'].emp_beta,
        'final_val_ppl': result['final_val_ppl'],
        'final_val_loss': result['final_val_loss'],
        'history': result['history'],
    }

with open('./checkpoints/emp_attention_sink_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print("Results saved to ./checkpoints/emp_attention_sink_results.json")

## 8. Conclusions

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

baseline_sink = all_results['baseline']['history']['first_token_sink'][-1]
sigmoid_sink = all_results.get('sigmoid', {}).get('history', {}).get('first_token_sink', [0])[-1]
emp_sink = all_results.get('emp_beta1.0', {}).get('history', {}).get('first_token_sink', [0])[-1]

print(f"""
1. ATTENTION SINK EMERGENCE:
   - Baseline (softmax): Sink strength = {baseline_sink:.4f}
   - Sigmoid (no norm):  Sink strength = {sigmoid_sink:.4f}
   - EMP (beta=1.0):     Sink strength = {emp_sink:.4f}

2. HYPOTHESIS VALIDATION:
   - Sigmoid attention eliminates attention sink (as expected from literature)
   - EMP during training {'reduces' if emp_sink < baseline_sink else 'does not significantly reduce'} attention sink
   
3. PERFORMANCE TRADE-OFF:
   - Baseline PPL: {all_results['baseline']['final_val_ppl']:.2f}
   - Sigmoid PPL:  {all_results.get('sigmoid', {}).get('final_val_ppl', 0):.2f}
   - EMP PPL:      {all_results.get('emp_beta1.0', {}).get('final_val_ppl', 0):.2f}

4. INTERPRETATION:
   - Attention sink emerges because softmax forces attention to sum to 1
   - First tokens become "garbage collectors" for unused attention mass
   - EMP pruning affects this dynamic by removing low-importance scores
   - The effect depends on beta: lower beta = more aggressive pruning
""")

print("\n" + "="*80)
print("OUTPUTS")
print("="*80)
print("""
- ./checkpoints/emp_attention_sink_training.png (training curves)
- ./checkpoints/emp_attention_sink_results.json (full results)
- ./checkpoints/attention_pattern_*.png (attention heatmaps)
""")